In [18]:
# ==========================================
# 9. COMPREHENSIONS & GENERATORS (Selected: Q117)
# ==========================================
# Question 117:
# Write a generator function csv_reader_gen(filename) that reads a CSV file line by line and yields one 
# parsed dict (header -> value) per row without loading the entire file into memory. Demonstrate processing 
# a CSV of at least 500 rows without storing them all at once.
#
# Sample Input:  filename = 'sample_500_rows.csv'
# Sample Output: {'id': '1', 'name': 'Item_1', 'val': '100'}
import csv
def csv_reader_gen(filename):
    with open(filename,'r',encoding="utf-8") as f:
        data = csv.reader(f)
        head = next(data)
        for r in data:
            yield dict(zip(head,r))

if __name__ == '__main__':
    # Test Question 117
    # Note: Create a dummy CSV file to test reading line-by-line
    test_filename = r"C:\Work\Python\Assignment 2.1\Dataset\Comp&generator\sample.csv"
    with open(test_filename, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['id', 'name', 'val'])
        for idx in range(1, 505):
            writer.writerow([idx, f'Item_{idx}', idx * 100])

    gen = csv_reader_gen(test_filename)
    print("Q117 Output:", next(gen) if gen else None)

Q117 Output: {'id': '1', 'name': 'Item_1', 'val': '100'}


In [28]:
# ==========================================
# 9. COMPREHENSIONS & GENERATORS (Selected: Q118)
# ==========================================
# Question 118:
# Using only comprehensions (no for loops), write a function that takes a list of sentences and builds a 
# word-frequency dictionary, ignoring stop words (provide a hardcoded list of at least 20 common stop 
# words) and words shorter than 3 characters.
#
# Sample Input:  sentences = ["Python is awesome and fun.", "Learning Python with fun examples is great!"]
# Sample Output: {'python': 2, 'awesome': 1, 'fun': 2, 'learning': 1, 'examples': 1, 'great': 1}

import string
def build_word_frequency(sentences):
    stop_words = {
        "the", "is", "and", "a", "an", "of",
        "to", "in", "on", "for", "with", "as",
        "at", "by", "from", "or", "but", "are",
        "was", "were", "this", "that"
    }
    res = [
        word.strip(string.punctuation).lower()
        for sent in sentences
        for word in sent.split()
        if len(word.strip(string.punctuation)) >= 3 and
        word.strip(string.punctuation).lower() not in stop_words
    ]
    return {x:res.count(x) for x in res}
if __name__ == '__main__':
    # Test Question 118
    sample_sentences = [
        "Python is awesome and fun.", 
        "Learning Python with fun examples is great!"
    ]
    print("Q118 Output:", build_word_frequency(sample_sentences))

Q118 Output: {'python': 2, 'awesome': 1, 'fun': 2, 'learning': 1, 'examples': 1, 'great': 1}


In [33]:
# ==========================================
# 9. COMPREHENSIONS & GENERATORS (Selected: Q119)
# ==========================================
# Question 119:
# Write a generator class InfiniteRange that mimics range() but has no upper limit. It should support 
# start, step, and optional stop. Implement __iter__ and __next__. Also implement a filter() method that 
# returns a new generator yielding only elements that pass a predicate.
#
# Sample Input:  InfiniteRange(start=10, step=2, stop=20).filter(lambda x: x % 3 == 0)
# Sample Output: [12, 18]

class InfiniteRange:
    def __init__(self, start=0, step=1, stop=None):
        self.start = start
        self.step = step
        self.stop = stop
        self.curr = start

    def __iter__(self):
        return self 

    def __next__(self):
        if self.stop is not None and self.curr >= self.stop:
            raise StopIteration
        val = self.curr
        self.curr += self.step
        return val


    def filter( self, p):
        def gen():
            for i in self:
                if p(i): yield i
        return gen()

if __name__ == '__main__':
    # Test Question 119
    rng = InfiniteRange(start=10, step=2, stop=20)
    filtered_gen = rng.filter(lambda x: x % 3 == 0) if hasattr(rng, 'filter') else None
    print("Q119 Output:", list(filtered_gen) if filtered_gen else None)

Q119 Output: [12, 18]


In [ ]:
# ==========================================
# 9. COMPREHENSIONS & GENERATORS (Selected: Q120)
# ==========================================
# Question 120:
# Write a function matrix_comprehension(n) that uses a single nested comprehension to generate an 
# n×n matrix where M[i][j] = gcd(i+1, j+1). Print it in a formatted grid. Then use a generator expression to 
# compute the sum of all elements.
#
# Sample Input:  n = 4
# Sample Output: ([[1, 1, 1, 1], [1, 2, 1, 2], [1, 1, 3, 1], [1, 2, 1, 4]], 24)


def gcd(a,b):
    return a if b==0 else gcd(b,a%b)

def matrix_comprehension(n):
    m = [[gcd(i+1,j+1) for i in range(n)] for j in range(n) ]
    s = sum(m[i][j] for i in range(n) for j in range(n))
    return (m,s)
if __name__ == '__main__':
    # Test Question 120
    print("Q120 Output:", matrix_comprehension(4))

Q120 Output: ([[1, 1, 1, 1], [1, 2, 1, 2], [1, 1, 3, 1], [1, 2, 1, 4]], 24)


In [ ]:
# ==========================================
# 9. COMPREHENSIONS & GENERATORS (Selected: Q121)
# ==========================================
# Question 121:
# Write a coroutine (a generator that uses send()) called running_stats() that accepts numbers one at 
# a time via send() and yields a tuple (count, mean, variance) after each number using Welford's online 
# algorithm. Demonstrate with at least 10 values.
#
# Sample Input:  send numbers 1 through 10 sequentially
# Sample Output: (10, 5.5, 9.17)

def running_stats():
    count = 0
    mean = 0
    M2 = 0

    while True:
        value = yield

        count += 1
        delta = value - mean
        mean += delta / count
        delta2 = value - mean
        M2 += delta * delta2
        variance = M2 / count

        yield (count, mean, variance)
if __name__ == '__main__':

    stats_coroutine = running_stats()
    next(stats_coroutine)
    last_stats = None           
    for val in range(1, 11):
        last_stats = stats_coroutine.send(val)
        next(stats_coroutine)

    print(
        "Q121 Output:",
        (
            last_stats[0],
            last_stats[1],
            round(last_stats[2], 2)
        )
    )

Q121 Output: (10, 5.5, 8.25)
